In [ ]:

import mlflow


import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

In [ ]:
mlflow.set_experiment("MLflow Quickstart")

mlflow.set_tracking_uri(uri="https://localhost/5000")
# Enable autologging for scikit-learn
mlflow.sklearn.autolog()

In [ ]:
# Load the Excel file
df = pd.read_excel('data/Telco_customer_churn.xlsx', engine='openpyxl')

# Display the first few rows
df.head()

# Fjerner mellomrom i alle kolonnenavn og gjør dem lettere å jobbe med
df.columns = [c.replace(' ', '') for c in df.columns]

df.columns

In [ ]:
df.columns

In [ ]:
# Sjekk fordelingen før vi gjør noe mer
print("Fordeling i originaldata:\n", df['ChurnLabel'].value_counts())

# Konverter y
y = df['ChurnLabel'].map({'Yes': 1, 'No': 0})

# Sjekk y etter konvertering
print("Fordeling i y etter mapping:\n", y.value_counts())

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(subset=['TotalCharges'], inplace=True)

# Sett stilen for plotting
sns.set_theme(style="whitegrid")

# --- VISUALISERING 1: KONTRAKSTYPE (Kategorisk) ---
plt.figure(figsize=(10, 5))
sns.countplot(x='Contract', hue='ChurnLabel', data=df, palette='viridis')
plt.title('Churn fordelt på Kontraktstype')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(df[df['ChurnLabel'] == 'No']['TenureMonths'], label='Blir værende', fill=True)
sns.kdeplot(df[df['ChurnLabel'] == 'Yes']['TenureMonths'], label='Slutter (Churn)', fill=True)
plt.xlabel('Måneder som kunde (Tenure)')
plt.title('Sannsynlighet for Churn basert på ansiennitet')
plt.legend()
plt.show()

In [ ]:
y = df['ChurnLabel'].apply(lambda x: 1 if x == 'Yes' else 0)

# 2. Definer X (Features) - dropp ID og selve målet
drop_cols = ['CustomerID', 'Count', 'Country', 'State', 'City', 'ZipCode', 
             'LatLong', 'Latitude', 'Longitude', 'ChurnLabel', 'ChurnValue', 
             'ChurnScore', 'CLTV', 'ChurnReason']

X = df.drop(columns=drop_cols)

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:

classifier = RandomForestClassifier(class_weight='balanced', random_state=42)
classifier.fit(X_train, y_train.values)
y_pred = classifier.predict(X_test)

In [ ]:
print(pd.Series(y_pred).value_counts())

print("Faktiske verdier i testsettet:", pd.Series(y_test).value_counts())
print("Modellens gjetninger:", pd.Series(y_pred).value_counts())

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')

conf_matrix = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='g', cmap='Blues', cbar=False)

plt.title('Confusion Matrix Heatmap')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()